Regression specifications:
delegitimizing ~ 

In [24]:
import pandas as pd

df = pd.read_csv('full_data.csv')
df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length', 'gpt_title_frame', 'gpt_excerpt_primary_frame',
       'gpt_excerpt_secondary_frame', 'gpt_demands_present',
       'gpt_protester_quote', 'gpt_authority_quote', 'gpt_thematic_present',
       'frames_error', 'binary_error'],
      dtype='str')

processing

In [32]:
# validated topics
topics_keep = [
    "Accident",
    "Crime",
    "Culture",
    "Economy",
    "Education",
    "Entertainment",
    "Finance",
    "Health",
    "Immigration",
    "Legal",
    "Lifestyle",
    "Politics",
    "Safety",
    "Social Issues",
    "Technology",
    "War",
    "Weather"
]



# topics_keep = [
#     "War",
#     "Social Issues",
#     "Labor",
#     "Legal",
#     "Politics",
#     "Crime"
# ]

df["delegit"] = df["gpt_excerpt_primary_frame"].isin(["protester_violence", "disruption", "property_damage"]).astype(int)

# New indicator: primary OR secondary is delegitimizing
df["delegit_any"] = (
    df["gpt_excerpt_primary_frame"].isin(["protester_violence", "disruption", "property_damage"])
    | df["gpt_excerpt_secondary_frame"].isin(["protester_violence", "disruption", "property_damage"])
).astype(int)



sub_df = df[
    df["gpt-topic"].isin(topics_keep)
].copy()


print("Full N:", len(df))
print("Regression N:", len(sub_df))
print("Percent retained:", round(len(sub_df) / len(df) * 100, 1))

# Compare prevalence
df[["delegit", "delegit_any"]].agg(
    ["sum", "mean"]
).rename(index={"sum": "N", "mean": "Rate"})

Full N: 9283
Regression N: 7458
Percent retained: 80.3


,delegit,delegit_any
N,3968.000000,4926.000000
Rate,0.427448,0.530647


In [34]:
main_topics = [
    "War", "Social Issues",
    "Legal", "Politics", "Crime"
]

topic_counts = sub_df["gpt-topic"].value_counts()

sparse_topics = topic_counts[
    (topic_counts < 100) &
    (~topic_counts.index.isin(main_topics))
].index

sub_df["topic_grouped"] = sub_df["gpt-topic"].replace(
    {topic: "Other" for topic in sparse_topics}
)

sub_df["topic_grouped"].value_counts()

topic_grouped
Politics         4657
Legal             682
Social Issues     483
Crime             467
War               382
Education         187
Entertainment     183
Other             157
Immigration       141
Culture           119
Name: count, dtype: int64

In [12]:
df['delegit'].value_counts()

delegit
0    5315
1    3968
Name: count, dtype: int64

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

# -----------------------
# Variables
# -----------------------

binary_cols = [
    #"demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

LEAN = 'C(political_leaning, Treatment(reference="center"))'
TOPIC = 'C(topic_grouped, Treatment(reference="War"))'


# -----------------------
# Helper: readable logit output
# -----------------------

def logit_table(model):
    """
    Return odds ratios, confidence intervals, and p-values
    from a fitted statsmodels logistic regression.
    """
    
    conf = model.conf_int()
    
    out = pd.DataFrame({
        "Odds Ratio": np.exp(model.params),
        "CI Low": np.exp(conf[0]),
        "CI High": np.exp(conf[1]),
        "p-value": model.pvalues
    })
    
    return out.round(3)


# Quick checks
print("Full dataset N:", len(df))
print("Six-topic subset N:", len(sub_df))

print("\nPolitical leaning:")
print(df["political_leaning"].value_counts())

print("\nTopics in sub_df:")
print(sub_df["gpt-topic"].value_counts())

In [35]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


LEAN = 'C(political_leaning, Treatment(reference="center"))'
TOPIC = 'C(topic_grouped, Treatment(reference="War"))'

coverage_cols = [
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

formulas = {
    "M1: Lean": LEAN,
    "M2: Lean + Topic": f"{LEAN} + {TOPIC}",
    "M3: + Coverage": f"{LEAN} + {TOPIC} + {' + '.join(coverage_cols)}"
}

models = {}

for outcome in ["delegit", "delegit_any"]:
    models[outcome] = {}

    for name, rhs in formulas.items():
        model = smf.logit(
            f"{outcome} ~ {rhs}",
            data=sub_df
        ).fit(disp=False)

        models[outcome][name] = model

In [37]:
comparison = results.pivot(
    index=["Model", "Predictor"],
    columns="Outcome",
    values="OR"
)

comparison.round(3)

Outcome                                                              delegit  \
Model            Predictor                                                     
M1: Lean         C(political_leaning, Treatment(reference="cente...    0.967   
                 C(political_leaning, Treatment(reference="cente...    0.929   
                 C(political_leaning, Treatment(reference="cente...    1.710   
                 C(political_leaning, Treatment(reference="cente...    1.129   
                 Intercept                                             0.743   
M2: Lean + Topic C(political_leaning, Treatment(reference="cente...    0.935   
                 C(political_leaning, Treatment(reference="cente...    0.944   
                 C(political_leaning, Treatment(reference="cente...    1.661   
                 C(political_leaning, Treatment(reference="cente...    1.145   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.270   
                 C(topic_grouped, Treatment(reference="War"))[T....    0.864   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.025   
                 C(topic_grouped, Treatment(reference="War"))[T....    0.965   
                 C(topic_grouped, Treatment(reference="War"))[T....    0.721   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.117   
                 C(topic_grouped, Treatment(reference="War"))[T....    0.878   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.696   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.912   
                 Intercept                                             0.505   
M3: + Coverage   C(political_leaning, Treatment(reference="cente...    0.976   
                 C(political_leaning, Treatment(reference="cente...    0.964   
                 C(political_leaning, Treatment(reference="cente...    1.646   
                 C(political_leaning, Treatment(reference="cente...    1.170   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.161   
                 C(topic_grouped, Treatment(reference="War"))[T....    0.890   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.035   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.008   
                 C(topic_grouped, Treatment(reference="War"))[T....    0.770   
                 C(topic_grouped, Treatment(reference="War"))[T....    0.998   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.040   
                 C(topic_grouped, Treatment(reference="War"))[T....    1.701   
                 C(topic_grouped, Treatment(reference="War"))[T....    2.126   
                 Intercept                                             0.525   
                 gpt_authority_quote                                   1.822   
                 gpt_protester_quote                                   0.796   
                 gpt_thematic_present                                  0.492   

Outcome                                                              delegit_any  
Model            Predictor                                                        
M1: Lean         C(political_leaning, Treatment(reference="cente...        0.944  
                 C(political_leaning, Treatment(reference="cente...        0.970  
                 C(political_leaning, Treatment(reference="cente...        1.627  
                 C(political_leaning, Treatment(reference="cente...        1.101  
                 Intercept                                                 1.077  
M2: Lean + Topic C(political_leaning, Treatment(reference="cente...        0.910  
                 C(political_leaning, Treatment(reference="cente...        0.980  
                 C(political_leaning, Treatment(reference="cente...        1.593  
                 C(political_leaning, Treatment(reference="cente...        1.118  
                 C(topic_grouped, Treatment(re

In [31]:
# =========================================================
# DELEGITIMIZING FRAME — BINARY LOGISTIC REGRESSIONS
# =========================================================

binary_rhs = " + ".join(binary_cols)


# M1: Political leaning only — FULL DATASET
m1_full = smf.logit(
    f"delegit ~ {LEAN}",
    data=df
).fit()


# M2: Political leaning only — SIX-TOPIC SUBSET
# This is the correct comparison model for M3.
m2_sub = smf.logit(
    f"delegit ~ {LEAN}",
    data=sub_df
).fit()


# M3: Add protest topic — SIX-TOPIC SUBSET
m3_topic = smf.logit(
    f"delegit ~ {LEAN} + {TOPIC}",
    data=sub_df
).fit()


# M4: Add other coverage dimensions — SIX-TOPIC SUBSET
m4_full = smf.logit(
    f"delegit ~ {LEAN} + {TOPIC} + {binary_rhs}",
    data=sub_df
).fit()


# -----------------------
# Readable output
# -----------------------

print("\n=== MODEL 1: Political leaning only — ALL ARTICLES ===")
display(logit_table(m1_full))

print("\n=== MODEL 2: Political leaning only — TOPICS ===")
display(logit_table(m2_sub))

print("\n=== MODEL 3: + Protest topic — TOPICS ===")
display(logit_table(m3_topic))

print("\n=== MODEL 4: + Binary coverage dimensions — TOPICS ===")
display(logit_table(m4_full))

Optimization terminated successfully.
         Current function value: 0.671645
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.681208
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.674169
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.653517
         Iterations 5

=== MODEL 1: Political leaning only — ALL ARTICLES ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.625,0.555,0.704,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.998,0.814,1.224,0.985
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.965,0.844,1.104,0.607
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.945,1.686,2.244,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.160,0.972,1.384,0.100



=== MODEL 2: Political leaning only — TOPICS ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.743,0.649,0.850,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.967,0.774,1.208,0.766
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.929,0.799,1.081,0.343
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.710,1.458,2.006,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.129,0.925,1.379,0.234



=== MODEL 3: + Protest topic — TOPICS ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.505,0.392,0.649,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.935,0.747,1.170,0.557
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.944,0.809,1.100,0.460
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.661,1.412,1.954,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.145,0.936,1.401,0.189
"C(topic_grouped, Treatment(reference=""War""))[T.Crime]",1.270,0.959,1.680,0.095
"C(topic_grouped, Treatment(reference=""War""))[T.Culture]",0.864,0.556,1.343,0.515
"C(topic_grouped, Treatment(reference=""War""))[T.Education]",1.025,0.711,1.476,0.895
"C(topic_grouped, Treatment(reference=""War""))[T.Entertainment]",0.965,0.664,1.403,0.853
"C(topic_grouped, Treatment(reference=""War""))[T.Immigration]",0.721,0.475,1.096,0.125



=== MODEL 4: + Binary coverage dimensions — TOPICS ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.525,0.395,0.698,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.976,0.776,1.228,0.836
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.964,0.824,1.128,0.649
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.646,1.394,1.943,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.170,0.952,1.438,0.136
"C(topic_grouped, Treatment(reference=""War""))[T.Crime]",1.161,0.871,1.547,0.309
"C(topic_grouped, Treatment(reference=""War""))[T.Culture]",0.890,0.567,1.398,0.613
"C(topic_grouped, Treatment(reference=""War""))[T.Education]",1.035,0.712,1.503,0.857
"C(topic_grouped, Treatment(reference=""War""))[T.Entertainment]",1.008,0.688,1.477,0.968
"C(topic_grouped, Treatment(reference=""War""))[T.Immigration]",0.770,0.504,1.178,0.228


From M2 (among the six prevalent topics) we see that Right outlets, relative to the Center (reference), give statistically significant increase in odds of delegitimizing frames. From M3, this still holds true even when we control for topics; it is not merely that right outlets cover different topics more.

Interpreting the topics coefficients in M3: we control for political leaning, meaning that social issues having higher rate of delegit is not because social issues is associated with more conservative outlets. Relative to War: Politics and Social Issues all have stat. significant higher odds of having delegitimizing frame. 

Interpreting binaries: After controlling for outlet lean and topic, articles containing a protester quote have 20% lower odds of being delegit framed compared to articles that do not. Articles with an authority quote are 71% higher odds. Articles with thematic present are 52% lower odds to have delegit frame. 

what if we don't collapse to delegitimizing binary, and keep as dominant frame? 

In [ ]:
# =========================================================
# MULTINOMIAL LOGIT — PRIMARY EXCERPT FRAME
# =========================================================

def prepare_multinomial(df, include_topic=False):
    
    cols = [
        "gpt_excerpt_primary_frame",
        "political_leaning"
    ]
    
    if include_topic:
        cols.append("gpt-topic")
    
    temp = df[cols].dropna().copy()

    # Outcome as categorical codes
    y_cat = pd.Categorical(temp["gpt_excerpt_primary_frame"])
    y = y_cat.codes

    # Political leaning dummies
    # Explicitly set Center as reference by dropping it
    lean_dummies = pd.get_dummies(
        temp["political_leaning"],
        prefix="lean",
        dtype=float
    )

    if "lean_center" in lean_dummies.columns:
        lean_dummies = lean_dummies.drop(columns="lean_center")

    X_parts = [lean_dummies]

    # Topic dummies if requested
    if include_topic:
        topic_dummies = pd.get_dummies(
            temp["gpt-topic"],
            prefix="topic",
            drop_first=True,
            dtype=float
        )
        
        X_parts.append(topic_dummies)

    X = pd.concat(X_parts, axis=1)
    X = sm.add_constant(X)

    return y, X, list(y_cat.categories)


# -----------------------
# Model A: leaning only, full dataset
# -----------------------

y_a, X_a, frame_categories_a = prepare_multinomial(
    df,
    include_topic=False
)

mn_frame_full = sm.MNLogit(y_a, X_a).fit()


# -----------------------
# Model B: leaning + topic, six-topic subset
# -----------------------

y_b, X_b, frame_categories_b = prepare_multinomial(
    sub_df,
    include_topic=True
)

mn_frame_topic = sm.MNLogit(y_b, X_b).fit()


print("\n=== MULTINOMIAL: POLITICAL LEANING ONLY ===")
print("Frame categories:")
print(dict(enumerate(frame_categories_a)))
print(mn_frame_full.summary())


print("\n=== MULTINOMIAL: POLITICAL LEANING + TOPIC ===")
print("Frame categories:")
print(dict(enumerate(frame_categories_b)))
print(mn_frame_topic.summary())

In [23]:
# =========================================================
# BINARY COVERAGE FEATURES
# Each outcome ~ political leaning
#              vs.
# Each outcome ~ political leaning + topic
# =========================================================

binary_models = {}
binary_results = {}

for outcome in binary_cols:
    
    # Same six-topic sample for both models
    temp = sub_df.dropna(
        subset=[outcome, "political_leaning", "gpt-topic"]
    ).copy()
    
    
    # Model A: Political leaning only
    model_lean = smf.logit(
        f"{outcome} ~ {LEAN}",
        data=temp
    ).fit(disp=False)
    
    
    # Model B: Political leaning + topic
    model_topic = smf.logit(
        f"{outcome} ~ {LEAN} + {TOPIC}",
        data=temp
    ).fit(disp=False)
    
    
    binary_models[outcome] = {
        "lean_only": model_lean,
        "lean_plus_topic": model_topic
    }
    
    binary_results[outcome] = {
        "lean_only": logit_table(model_lean),
        "lean_plus_topic": logit_table(model_topic)
    }


# -----------------------
# Display results
# -----------------------

for outcome in binary_cols:
    
    print("\n" + "=" * 70)
    print(outcome.upper())
    print("=" * 70)
    
    print("\nPolitical leaning only:")
    display(binary_results[outcome]["lean_only"])
    
    print("\nPolitical leaning + protest topic:")
    display(binary_results[outcome]["lean_plus_topic"])


GPT_PROTESTER_QUOTE

Political leaning only:


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.753,0.660,0.860,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.117,0.899,1.388,0.319
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",1.619,1.395,1.880,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.299,1.108,1.523,0.001
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.169,0.958,1.428,0.124



Political leaning + protest topic:


,Odds Ratio,CI Low,CI High,p-value
Intercept,2.537,2.041,3.154,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.180,0.943,1.476,0.148
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",1.700,1.455,1.985,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.484,1.258,1.751,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.233,1.003,1.515,0.047
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",0.121,0.092,0.158,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",0.147,0.115,0.188,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",0.281,0.231,0.341,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",0.255,0.197,0.332,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",0.295,0.224,0.388,0.000



GPT_AUTHORITY_QUOTE

Political leaning only:


...

,Odds Ratio,CI Low,CI High,p-value
Intercept,2.071,1.667,2.572,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.992,0.760,1.296,0.956
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",1.019,0.849,1.224,0.841
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.301,1.061,1.594,0.011
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.119,0.873,1.435,0.376
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",1.524,1.160,2.003,0.002
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",1.986,1.541,2.559,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",2.037,1.699,2.442,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",1.709,1.290,2.264,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",1.841,1.359,2.493,0.000



GPT_THEMATIC_PRESENT

Political leaning only:


,Odds Ratio,CI Low,CI High,p-value
Intercept,1.554,1.359,1.778,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.240,0.990,1.553,0.062
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.947,0.814,1.102,0.480
"C(political_leaning, Treatment(reference=""center""))[T.right]",0.900,0.766,1.057,0.199
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.089,0.888,1.336,0.414



Political leaning + protest topic:


,Odds Ratio,CI Low,CI High,p-value
Intercept,1.348,1.110,1.637,0.003
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.223,0.975,1.534,0.081
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.925,0.794,1.078,0.319
"C(political_leaning, Treatment(reference=""center""))[T.right]",0.828,0.703,0.976,0.024
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.081,0.880,1.327,0.460
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",0.873,0.688,1.109,0.267
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",0.857,0.691,1.063,0.160
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",1.244,1.054,1.468,0.010
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",2.170,1.681,2.802,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",1.349,1.041,1.748,0.024


Reinforcing the descriptive table of binaries X topic, we see that topics have stat. significant lower odds of protester quotes than labor (reference), but stat. significantly higher odds of authority quotes relative to labor protest articles.